# animations — TRELLIS single-image → 3D (best single-photo quality)

**TRELLIS** turns **one** photo into a detailed, **textured** 3D model — far
sharper than TripoSR. Same one-photo workflow you already use.

**Steps:** Runtime → Change runtime type → **GPU (T4 works; L4/A100 faster)**,
then Runtime → **Run all**. When the last cell prints a `gradio.live` URL, paste
it into the Studio's **Connect a generator** field and Generate with just the
**Front** photo.

> ⚠️ **Heaviest notebook by far.** It builds custom CUDA extensions and pins
> torch, so the first run takes **~15–20 min** and is the most likely to hit a
> version snag. If a cell turns red, copy the error — it's almost always one
> dependency wheel to adjust. It produces **textured** models (real colors).
> The `gradio.live` link changes every session.


In [ ]:
# 1) Pin torch for reproducible CUDA wheels, then install TRELLIS + extensions.
# (~15-20 min the first time — downloads torch, builds CUDA rasterizers.)
import os
os.environ["ATTN_BACKEND"] = "xformers"   # skip flash-attn (its build often fails on Colab)
os.environ["SPCONV_ALGO"] = "native"

%cd /content
!rm -rf TRELLIS
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
%cd /content/TRELLIS

# Deterministic torch so kaolin/xformers/spconv wheels line up (cu121).
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

# Core Python deps (from TRELLIS setup.sh "basic" + utils3d):
!pip install -q pillow imageio imageio-ffmpeg tqdm easydict opencv-python-headless \
    scipy ninja rembg onnxruntime trimesh xatlas pyvista pymeshfix igraph transformers
!pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8

# GPU building blocks matched to torch 2.4.0 / cu121:
!pip install -q xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q spconv-cu120
!pip install -q kaolin==0.16.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html

# CUDA rasterizers TRELLIS renders/textures with (compiled here):
!pip install -q git+https://github.com/NVlabs/nvdiffrast.git
!pip install -q git+https://github.com/JeffreyXiang/diffoctreerast.git
!pip install -q ./extensions/mip-splatting/submodules/diff-gaussian-rasterization/

# Studio-compatible server stack (same pins as the other notebooks):
!pip install -q -U "gradio==4.44.1" "gradio-client==1.3.0" "huggingface_hub==0.25.2" "pydantic==2.10.6"
print("Install finished. If nothing above is red, run the next cell.")


In [ ]:
# 2) Single-image TRELLIS app the Studio drives (api_name="generate"), + share link.
import os, tempfile
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
%cd /content/TRELLIS

import gradio as gr
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

print("Loading TRELLIS-image-large (first time downloads a few GB)...")
pipe = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
pipe.cuda()

def generate(image):
    if image is None:
        raise gr.Error("Upload a device photo.")
    # TRELLIS removes the background itself (preprocess_image=True by default).
    outputs = pipe.run(image.convert("RGB"), seed=1, formats=["gaussian", "mesh"])
    glb = postprocessing_utils.to_glb(
        outputs["gaussian"][0], outputs["mesh"][0], simplify=0.95, texture_size=1024,
    )
    path = os.path.join(tempfile.mkdtemp(), "model.glb")
    glb.export(path)
    return path

with gr.Blocks(title="animations - TRELLIS single-image to 3D") as demo:
    gr.Markdown("## animations - TRELLIS single-image to 3D\nDriven by the Studio; you can also test here directly.")
    img = gr.Image(type="pil", label="Device photo")
    out = gr.Model3D(label="Result (.glb)")
    gr.Button("Generate", variant="primary").click(generate, img, out, api_name="generate")

print("Watch for:  Running on public URL: https://XXXX.gradio.live")
demo.queue(max_size=4).launch(share=True)


## Use it
1. Copy the `https://XXXX.gradio.live` URL printed above.
2. Open the Studio: **zurlazar.github.io/animations/studio.html**
3. Paste the URL into **Connect a generator** -> **Save**.
4. Put your photo in the **Front** slot -> **Generate 3D model** (other slots are
   ignored by this single-image model).
5. **Download .glb** when you like the result.

Quality tips: a clean, single, sharp, well-lit product shot on a plain
background still helps a lot — but TRELLIS is far more forgiving than TripoSR.
